# Notebook 1 — The Network and Degree
**YTS+ DSEB 2026 · Project A · Harry Potter**

---

**Before you open this notebook**, each student writes down independently:
1. The 5 characters you believe the story could least afford to lose.
2. Your prediction: if Harry were removed entirely, who becomes the most important character?

Paper. Sealed. Not discussed with your team. You will return to these at the end of the project.

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
import os

os.makedirs('images', exist_ok=True)

DATA = 'data/'
BOOK_NAMES = ['Stone', 'Secrets', 'Azkaban', 'Fire', 'Phoenix', 'Prince', 'Hallows']

print('Ready.')

---
## Part 1 — What's in the data?

Before building anything, look at the raw data. Each row is a pair of characters who appear within 14 words of each other in the text. The `weight` is how many times that happened.

In [ ]:
# Load Book 3 — Prisoner of Azkaban
df3 = pd.read_csv(DATA + 'hp_book3_edges.csv')

print(f'Rows (character pairs): {len(df3)}')
print(f'Columns: {df3.columns.tolist()}')
print()
df3.head(10)

In [ ]:
# What does the weight distribution look like?
print(df3['weight'].describe())
print(f'\nMax weight pair:')
print(df3.sort_values('weight', ascending=False).head(1).to_string(index=False))

In [ ]:
# Most pairs appear together only a few times — a few appear hundreds of times
plt.figure(figsize=(9, 4))
plt.hist(df3['weight'], bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Co-appearance count (edge weight)')
plt.ylabel('Number of pairs')
plt.title('Book 3 — Distribution of co-appearance counts')
plt.tight_layout()
plt.savefig('images/book3_weight_hist.png', dpi=150)
plt.show()

print(f'Pairs appearing together only once: {(df3["weight"] == 1).sum()}')
print(f'Pairs appearing 10+ times: {(df3["weight"] >= 10).sum()}')
print(f'Pairs appearing 50+ times: {(df3["weight"] >= 50).sum()}')

In [ ]:
# Top 10 pairs by weight — who does Rowling place together most?
top_pairs = df3.sort_values('weight', ascending=False).head(10)
print('Top 10 most frequent pairs in Book 3:')
print(top_pairs.to_string(index=False))

**Crabbe appears with Goyle 95+ times.** That means Rowling wrote them in the same moment, in the same scene, 95 times across one book. Is that friendship — or does Rowling treat them as a single unit?

This is the kind of question the network will make precise.

In [ ]:
# Compare all 7 books: how many pairs, how many unique characters?
t0 = time.time()

print(f'{"Book":<6} {"Name":<10} {"Pairs":>8} {"Characters":>12}')
print('-' * 40)

for i in range(1, 8):
    df = pd.read_csv(DATA + f'hp_book{i}_edges.csv')
    chars = set(df['source']) | set(df['target'])
    print(f'{i:<6} {BOOK_NAMES[i-1]:<10} {len(df):>8} {len(chars):>12}')

print(f'\nLoaded all 7 books in {time.time()-t0:.2f}s')

**Notice anything?** The network does not grow monotonically. It shrinks, then explodes, then stabilises. That is a trace of Rowling's narrative decisions — and you will explain it in Part 4.

---
## Part 2 — Build the network

**Before building:** Which 5 characters do you predict will have the highest weighted degree in Book 3?

**My top 5 predictions (Book 3 weighted degree):**

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Build the Book 3 graph
t0 = time.time()

G3 = nx.from_pandas_edgelist(df3, 'source', 'target', edge_attr='weight')

print(f'Graph built in {time.time()-t0:.3f}s')
print(f'Characters (nodes): {G3.number_of_nodes()}')
print(f'Pairs who appear together (edges): {G3.number_of_edges()}')

In [ ]:
# Quick exploration: who has the most connections?
# (unweighted — just counts unique partners)
by_partners = sorted(G3.degree(), key=lambda x: x[1], reverse=True)

print('Top 5 by number of unique characters they appear with:')
for name, deg in by_partners[:5]:
    print(f'  {name}: {deg} unique partners')

print(f'\nAverage: {sum(d for _, d in G3.degree()) / G3.number_of_nodes():.1f} unique partners per character')

---
## Part 3 — Weighted degree

**Unweighted degree** counts how many unique characters you appear with.

**Weighted degree** sums up the weights — how many total co-appearances across all your partners.

Weighted degree tells you how much Rowling used this character across scenes, not just how many different people they appeared with.

In [ ]:
# Weighted degree for every character
t0 = time.time()

degree3 = dict(G3.degree(weight='weight'))

print(f'Computed in {time.time()-t0:.4f}s')  # instant — note this for contrast later
print()

top15 = sorted(degree3.items(), key=lambda x: x[1], reverse=True)[:15]

print('Top 15 characters by weighted degree (Book 3):')
for i, (name, deg) in enumerate(top15):
    print(f'  {i+1:>2}. {name:<30} {deg}')

In [ ]:
names = [n for n, _ in top15]
values = [v for _, v in top15]

plt.figure(figsize=(9, 6))
plt.barh(names[::-1], values[::-1], color='steelblue')
plt.xlabel('Weighted degree (total co-appearances)')
plt.title('Book 3 — Top 15 characters by weighted degree')
plt.tight_layout()
plt.savefig('images/book3_degree.png', dpi=150)
plt.show()

**Check your sealed predictions.**

- Which were right? Which were wrong?
- Where does your intuition agree with degree? Where does it diverge?

**Write here:**

*Your answer.*

---

**Important:** Degree measures how broadly Rowling placed this character across scenes. A character with high degree appeared alongside many different people, many times. That is not the same as being important to the plot. A character can be everywhere without being load-bearing.

---
## Part 4 — Network expansion across 7 books

**Before computing:** Predict the shape of the curve.
- When does the network grow fastest?
- Does it ever shrink?
- Which book has the most characters?

**My predictions:**

- Grows fastest in Book: ___  because: ___
- Shrinks? Yes / No — which books: ___
- Most characters in Book: ___

In [ ]:
# Count nodes and edges for each individual book
t0 = time.time()

node_counts = []
edge_counts = []

for i in range(1, 8):
    df = pd.read_csv(DATA + f'hp_book{i}_edges.csv')
    G = nx.from_pandas_edgelist(df, 'source', 'target', edge_attr='weight')
    node_counts.append(G.number_of_nodes())
    edge_counts.append(G.number_of_edges())
    print(f'Book {i} ({BOOK_NAMES[i-1]}): {G.number_of_nodes()} characters, {G.number_of_edges()} pairs')

print(f'\nTotal time: {time.time()-t0:.2f}s')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = range(1, 8)

ax1.plot(x, node_counts, 'o-', color='steelblue', linewidth=2, markersize=8)
for i, n in enumerate(node_counts):
    ax1.annotate(str(n), (i+1, n), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels(BOOK_NAMES, rotation=30, ha='right')
ax1.set_ylabel('Number of characters')
ax1.set_title('Characters per book')
ax1.grid(alpha=0.3)

ax2.plot(x, edge_counts, 'o-', color='tomato', linewidth=2, markersize=8)
for i, n in enumerate(edge_counts):
    ax2.annotate(str(n), (i+1, n), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(BOOK_NAMES, rotation=30, ha='right')
ax2.set_ylabel('Number of interacting pairs')
ax2.set_title('Pairs per book')
ax2.grid(alpha=0.3)

plt.suptitle("Rowling's world — per book", fontsize=13)
plt.tight_layout()
plt.savefig('images/network_expansion.png', dpi=150)
plt.show()

**Write one paragraph below.** Use your knowledge of the books to explain each inflection point.

Start here:
- Why does the network shrink from Book 1 (122 nodes) to Book 3 (90 nodes)?
- Why does it explode in Book 4?
- Why does it peak in Book 5?

Name three specific plot decisions by Rowling that are visible in this data.

**Your paragraph:**

*Write here.*

---
## What to bring to the next session

- Your sealed predictions (keep them sealed until the end of the project)
- Your paragraph on network expansion
- One question the data raised that you can't answer yet

**Next:** Notebook 2 — Community Detection. The algorithm will partition the network into groups using only edge weights. No plot knowledge. We will ask: does it find the factions Rowling built?